# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils.UVA as UVA
import utils.MVA as MVA
import utils.PCA as PCA

from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.decomposition import PCA as skPCA
from sklearn.compose import ColumnTransformer

from sklearn.cluster import KMeans, DBSCAN, MiniBatchKMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples

from sklearn.metrics.cluster import adjusted_rand_score

In [2]:
df = pd.read_csv('data/RFM_base.csv')
df.set_index('customer_unique_id', inplace=True)

df

,recency,frequency,monetary,max_order_date
customer_unique_id,,,,
871766c5855e863f6eccc05f988b23cb,399,1,58.90,2017-09-13
eb28e67c4c0b83846050ddfb8a35d051,394,2,252.78,2017-09-18
3818d81c6709e39d06b2738a8d3a2474,276,1,199.00,2018-01-14
af861d436cfc08b2c2ddefd0ba074622,70,1,12.99,2018-08-08
64b576fb70d441e8f1b2d7d446e483c5,620,1,199.90,2017-02-04
...,...,...,...,...
0c9aeda10a71f369396d0c04dce13a64,177,1,299.99,2018-04-23
0da9fe112eae0c74d3ba1fe16de0988b,95,1,350.00,2018-07-14
cd79b407828f02fdbba457111c38e4c4,359,1,99.90,2017-10-23


In [3]:
offset = -1
n_periods_max = 24
n_periods_min = 8

#list_offsets = [i for i in range(n_offset, n_periods+n_offset, n_offset)]
list_offsets = list(range(n_periods_max, n_periods_min, offset))
list_offsets

[24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9]

In [4]:
min_date = pd.to_datetime(df['max_order_date'].min())
max_date = pd.to_datetime(df['max_order_date'].max())

print(min_date.strftime('%Y-%m-%d'))
print(max_date.strftime('%Y-%m-%d'))

2016-09-04
2018-10-16


In [5]:
n_period_sp = 5
limit_date = (max_date - pd.DateOffset(weeks=list_offsets[n_period_sp])).strftime('%Y-%m-%d')

print (f"Date max {max_date.strftime('%Y-%m-%d')} après retrait de {n_period_sp} semaines : {limit_date}")

Date max 2018-10-16 après retrait de 5 semaines : 2018-06-05


In [6]:
dfs = {}

#list_n_df
#len_offsets = len(list_offsets)
#offsets_count = list(range(1, len_offsets, 1))
n = 1

for i in list_offsets:
    subset = df[df['max_order_date'] <= (max_date - pd.DateOffset(weeks=i)).strftime('%Y-%m-%d')]
    #dfs[f'df_{i}'] = subset
    dfs[f'df_{n}'] = subset
    n+=1

print(dfs.keys())

dict_keys(['df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6', 'df_7', 'df_8', 'df_9', 'df_10', 'df_11', 'df_12', 'df_13', 'df_14', 'df_15', 'df_16'])


In [7]:
for name, df in dfs.items():
    print(f'{name}: shape {df.shape}, max date {df.max_order_date.max()}')

df_1: shape (70279, 4), max date 2018-05-01
df_2: shape (72213, 4), max date 2018-05-08
df_3: shape (74169, 4), max date 2018-05-15
df_4: shape (75661, 4), max date 2018-05-22
df_5: shape (76489, 4), max date 2018-05-29
df_6: shape (77672, 4), max date 2018-06-05
df_7: shape (79210, 4), max date 2018-06-12
df_8: shape (80644, 4), max date 2018-06-19
df_9: shape (82089, 4), max date 2018-06-26
df_10: shape (83398, 4), max date 2018-07-03
df_11: shape (84448, 4), max date 2018-07-10
df_12: shape (85605, 4), max date 2018-07-17
df_13: shape (87348, 4), max date 2018-07-24
df_14: shape (88992, 4), max date 2018-07-31
df_15: shape (91157, 4), max date 2018-08-07
df_16: shape (92980, 4), max date 2018-08-14


In [8]:
# drop max_order_date column
for name, df in dfs.items():
    df = df.drop('max_order_date', axis=1)
    dfs[name] = df

for name, df in dfs.items():
    print(f'{name}: {df.shape}')
    #print()

df_1: (70279, 3)
df_2: (72213, 3)
df_3: (74169, 3)
df_4: (75661, 3)
df_5: (76489, 3)
df_6: (77672, 3)
df_7: (79210, 3)
df_8: (80644, 3)
df_9: (82089, 3)
df_10: (83398, 3)
df_11: (84448, 3)
df_12: (85605, 3)
df_13: (87348, 3)
df_14: (88992, 3)
df_15: (91157, 3)
df_16: (92980, 3)


In [9]:
dfs['df_1']

,recency,frequency,monetary
customer_unique_id,,,
871766c5855e863f6eccc05f988b23cb,399,1,58.90
eb28e67c4c0b83846050ddfb8a35d051,394,2,252.78
3818d81c6709e39d06b2738a8d3a2474,276,1,199.00
64b576fb70d441e8f1b2d7d446e483c5,620,1,199.90
85c835d128beae5b4ce8602c491bf385,520,1,21.90
...,...,...,...
6b42acb204802253acec6607ff3a9e0b,201,1,17.90
f736308cd9952b33b90b9fe94da9c8f5,355,1,220.00
0c9aeda10a71f369396d0c04dce13a64,177,1,299.99


# train kmeans datasets


In [10]:
skewness = dfs['df_1'].select_dtypes(include=np.number).skew()
treshold_skew = 1.0
cols_skewed = skewness[skewness > treshold_skew].index.tolist()
cols_unskewed = skewness[skewness < treshold_skew].index.tolist()

print("Colonnes numériques avec une skewness supérieure à la limite :")
print(cols_skewed)
print(cols_unskewed)


Colonnes numériques avec une skewness supérieure à la limite :
['frequency', 'monetary']
['recency']


In [11]:
def customTransform(df, cols_skewed, cols_unskewed):
    # sélection colonnes avec skewness élevée :
    
    # Création du ColumnTransformer
    transformer = ColumnTransformer([
        #('minmax', MinMaxScaler(), cols_skewed),
        ('power', PowerTransformer(), cols_skewed),
        ('standard', StandardScaler(), cols_unskewed)
    ])
    
    X_std = pd.DataFrame(transformer.fit_transform(df), columns=df.columns)
    return X_std

In [12]:
# drop max_order_date column
X_stds = {}

for name, df in dfs.items():
    df = customTransform(df, cols_skewed, cols_unskewed)
    name_split = name.split("_")
    X_stds[f'X_{name_split[1]}'] = df

In [13]:
X_stds.keys()

dict_keys(['X_1', 'X_2', 'X_3', 'X_4', 'X_5', 'X_6', 'X_7', 'X_8', 'X_9', 'X_10', 'X_11', 'X_12', 'X_13', 'X_14', 'X_15', 'X_16'])

In [14]:
for name, df in X_stds.items():
    print(f'{name}: {df.shape}')
    #print()

X_1: (70279, 3)
X_2: (72213, 3)
X_3: (74169, 3)
X_4: (75661, 3)
X_5: (76489, 3)
X_6: (77672, 3)
X_7: (79210, 3)
X_8: (80644, 3)
X_9: (82089, 3)
X_10: (83398, 3)
X_11: (84448, 3)
X_12: (85605, 3)
X_13: (87348, 3)
X_14: (88992, 3)
X_15: (91157, 3)
X_16: (92980, 3)


In [15]:
X_stds['X_1']

,recency,frequency,monetary
0,-6.938894e-18,-0.391222,0.377950
1,2.255141e-16,1.132828,0.338416
2,-6.938894e-18,0.895501,-0.594607
3,-6.938894e-18,0.900024,2.125391
4,-6.938894e-18,-1.527538,1.334694
...,...,...,...
70274,-6.938894e-18,-1.767002,-1.187630
70275,-6.938894e-18,0.995650,0.030044
70276,-6.938894e-18,1.299602,-1.377398
70277,-6.938894e-18,0.183284,0.061671


## Check training first dataset

In [ ]:
X_std = X_stds['X_1'].copy()

# Calcul du score de silhouette pour chaque valeur de n_clusters comprise entre 2 et 15
n_clusters_range = range(2, 16)
scores = []

columns = X_std.columns

for n_clusters in n_clusters_range:
    # Initialisation de KMeans avec kmeans++
    
    #kmeans = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, max_iter=300, random_state=42)
    #kmeans.fit(X_std)
    #labels = kmeans.labels_
    #score = silhouette_score(X_std, labels)
    #print(f'Kmeans sil score : {score}')
    #scores.append(score)

    # Initialisation de MiniBatchKMeans
    mbkmeans = MiniBatchKMeans(n_clusters=n_clusters, init='k-means++', n_init=10, max_iter=300, batch_size=100, random_state=42)
    mbkmeans.fit(X_std)
    labels_mb = mbkmeans.labels_
    score_mb = silhouette_score(X_std, labels_mb)
    print(f'BatchKmeans {n_clusters} clusters, sil score : {score_mb}')
    scores.append(score_mb)

# Tracé du graphique des scores de silhouette
plt.plot(n_clusters_range, scores)
plt.xlabel('Nombre de clusters')
plt.ylabel('Score de silhouette')
plt.title('Scores de silhouette en fonction du nombre de clusters')
plt.show()

# Sélection de la valeur de n_clusters qui donne le score de silhouette le plus élevé
best_score = max(scores)
best_index = scores.index(best_score)
best_n_clusters = best_index + 2

print(f"Le nombre optimal de clusters est : {best_n_clusters}")

BatchKmeans 2 clusters, sil score : 0.3733558169229127
BatchKmeans 3 clusters, sil score : 0.3722880255097268
BatchKmeans 4 clusters, sil score : 0.32772729727988686
BatchKmeans 5 clusters, sil score : 0.3272781482577685
BatchKmeans 6 clusters, sil score : 0.32543666289529166
BatchKmeans 7 clusters, sil score : 0.33597841712577786


In [ ]:
sorted_scores = np.argsort(scores)[::-1]
top_indices = sorted_scores[:5]

print("Les 5 plus grands score de silhouette sont :")

for indice in top_indices:
    print(f"{indice+2} clusters pour {scores[indice]} score de silhouette")

In [ ]:
inertia = {}
dict_kmeans = {}
for k in range(1,25):
    print(k)
    kmeans = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0
                   ).fit(X_std)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans

In [ ]:
plt.figure(figsize=(8,5))
plt.title('Kmeans : Comparaison de la somme des inerties en fonction du nombre de clusters')
sns.lineplot(x=list(inertia.keys()),
             y=list(inertia.values())
            )

### Train and compare through time

In [ ]:
clusters_num = 4

dict_Kmeans_X = {}

for name, df in X_stds.items():
    kmeans = KMeans(n_clusters=clusters_num, verbose=1, random_state=0).fit(df)
    dict_Kmeans_X[f'kmeans_{name}'] = kmeans

# Compute ARI

In [ ]:
scores = []
first_kmeans = dict_Kmeans_X['kmeans_X_1']

for name, kmeans in dict_Kmeans_X.items():
    X_name = name.replace("kmeans_", "")
    ari = adjusted_rand_score(kmeans.labels_, first_kmeans.predict(X_stds[X_name]))
    scores.append(ari)

#periodes = 

In [ ]:
scores

In [ ]:
scores_len = len(scores)
periodes = list(range(0, scores_len, 1))

In [ ]:
plt.plot(periodes, scores, marker='o')
plt.xlabel('Période')
plt.ylabel('Score ARI')
plt.title('Évolution du score ARI en fonction de la période')
plt.show()

In [ ]:
scores[0:13]

In [ ]:
scores[12] - scores[11]